# OCAPI Data API — Content assets (list + by ID)

Uses the **OCAPI Data API** (via `instance.ocapi`) with **OAuth
client-credentials** to read a content **library**:

- **list a folder's content** — `GET /libraries/{library_id}/folders/{folder_id}/content`, and
- **fetch one content asset** — `GET /libraries/{library_id}/content/{content_id}`

Responses are parsed into the **generated Pydantic types**
`b2c_tooling_sdk.clients.models.ocapi.ContentAssetResult` and `ContentAsset`.

> The OCAPI Data API has no `content_search`, so listing is done by reading a
> folder's content (start at the `root` folder). You must supply a **library id**:
> a site's private library is typically `<siteId>-Library`; a shared library uses
> its own id (e.g. `RefArchSharedLibrary`). Connection settings come from
> `../dw.json` (needs `clientId`, `clientSecret`).

In [ ]:
from pathlib import Path

from b2c_tooling_sdk import ResolveConfigOptions, resolve_config
from b2c_tooling_sdk.clients.models.ocapi import ContentAsset, ContentAssetResult

DW_JSON = Path("../dw.json").resolve()

config = await resolve_config(options=ResolveConfigOptions(config_path=str(DW_JSON)))
instance = config.create_b2c_instance()
print(f"OCAPI Data API on {instance.config.hostname}")


def local(value):
    """Pick a localized value from an OCAPI localized dict (prefers 'default')."""
    if isinstance(value, dict):
        return value.get("default") or next(iter(value.values()), None)
    return value


def text_of(markup):
    """Render a MarkupText (or plain value) to a short string."""
    if markup is None:
        return None
    return getattr(markup, "markup", None) or getattr(markup, "source", None) or str(markup)


def result_data(result, what):
    """Return parsed data on success; print a friendly message and return None otherwise."""
    if result.error is None and result.data is not None:
        return result.data
    status = result.response.status_code if result.response is not None else "?"
    print(f"{what}: HTTP {status} — {str(result.error)[:200]}")
    return None

In [ ]:
# --- Parameters (edit these) ---
LIBRARY_ID = ""        # REQUIRED, e.g. "RefArchSharedLibrary" or "<siteId>-Library"
FOLDER_ID = "root"     # folder to list; "root" is the top of the library
CONTENT_ID = ""        # blank -> use the first asset from the listing below
RESULT_LIMIT = 10      # max assets to list

## 1. List a folder's content (`GET /libraries/{library_id}/folders/{folder_id}/content`)

Parsed into the generated `ContentAssetResult` (its `hits` are `ContentAsset`s).

In [ ]:
assets = []
if not LIBRARY_ID:
    print("Set LIBRARY_ID first (e.g. 'RefArchSharedLibrary' or '<siteId>-Library').")
else:
    result = await instance.ocapi.get(
        "/libraries/{library_id}/folders/{folder_id}/content",
        {
            "params": {
                "path": {"library_id": LIBRARY_ID, "folder_id": FOLDER_ID},
                "query": {"start": 0, "count": RESULT_LIMIT, "select": "(**)"},
            }
        },
    )
    data = result_data(result, f"libraries/{LIBRARY_ID}/folders/{FOLDER_ID}/content")
    if data is not None:
        listing = ContentAssetResult.model_validate(data)  # <- generated type
        assets = listing.hits or []
        print(f"folder '{FOLDER_ID}' in '{LIBRARY_ID}': {listing.total} asset(s), showing {len(assets)}:")
        for a in assets:
            print(f"  • {(a.id or '?'):<28} {local(a.name) or '(no name)'}")
        if not assets:
            print("  (no assets — try a different FOLDER_ID or LIBRARY_ID)")

## 2. Content asset by ID (`GET /libraries/{library_id}/content/{content_id}`)

Uses `CONTENT_ID` if set, otherwise the first asset listed above. Parsed into the
generated `ContentAsset` model.

In [ ]:
content_id = CONTENT_ID or (assets[0].id if assets else None)
if not LIBRARY_ID or not content_id:
    print("No content id available — set LIBRARY_ID and CONTENT_ID (or ensure the listing returned assets).")
else:
    result = await instance.ocapi.get(
        "/libraries/{library_id}/content/{content_id}",
        {"params": {"path": {"library_id": LIBRARY_ID, "content_id": content_id}, "query": {"select": "(**)"}}},
    )
    data = result_data(result, f"libraries/{LIBRARY_ID}/content/{content_id}")
    if data is not None:
        asset = ContentAsset.model_validate(data)  # <- generated type
        print("id     :", asset.id)
        print("name   :", local(asset.name))
        print("online :", local(asset.online))
        print("desc   :", local(asset.description))
        body = text_of(local(asset.c_body))
        if body:
            print("body   :", body[:200].strip(), "...")